# HerdNet Minimal working example

In [ ]:
## Installation

In [4]:
!conda env create -n HerdNet -f ../environment.yml

Channels:
 - pytorch
 - defaults
 - conda-forge
Platform: osx-arm64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.5.0
    latest version: 25.11.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



scipy-1.16.0         | 22.7 MB   |                                       |   0% 
pandas-2.3.1         | 15.0 MB   |                                       |   0% 

numpy-base-2.4.1     | 6.9 MB    |                                       |   0% 


libopenblas-0.3.30   | 4.1 MB    |                                       |   0% 



libgfortran5-15.2.0  | 742 KB    |                                       |   0% 




llvm-openmp-20.1.8   | 327 KB    |                                       |   0% 





numexpr-2.14.1       | 193 KB    |                                       |   0% 






bottleneck-1.4.2     | 122 KB    |                                       |   0% 







numpy-2.4.1          | 20 KB     |    


CondaError: Run 'conda init' before 'conda activate'



In [ ]:
## Optional update if the environment changed
!conda env update --file environment.yml --prune


## (Optional) Install Active Learning Repository
This contains functions for Training Data Preparation, Geospatial Inference and Detection Deduplication.
TODO

In [ ]:
import sys
sys.path.append('./')

In [4]:
from pathlib import Path
Path("./").resolve()

PosixPath('/home/cwinkelmann/work/HerdNet/notebooks')

In [12]:
import pandas as pd
anno_path = Path("/home/cwinkelmann/work/software-data-engineering-mac/data/ISAID/train/herdnet_format.csv")

anno_path.is_file()



True

In [15]:
df = pd.read_csv(anno_path)
cls_dict = (df[['labels', 'species']]
            .drop_duplicates()
            .set_index('labels')['species']
            .to_dict())

print(cls_dict)

{3: 'Small_Vehicle', 2: 'Large_Vehicle', 4: 'plane', 1: 'storage_tank', 5: 'ship', 7: 'Harbor', 6: 'Swimming_pool', 8: 'tennis_court', 12: 'Bridge', 13: 'basketball_court', 11: 'baseball_diamond', 14: 'Roundabout', 15: 'Helicopter', 10: 'Soccer_ball_field', 9: 'Ground_Track_Field'}


In [19]:
import yaml

class_counts = df['labels'].value_counts()

# 2. Convert counts to frequencies (i.e., fraction of the total)
class_freqs = class_counts / class_counts.sum()

# 3. Compute inverse frequency for each class
class_weights_inv = 1.0 / class_freqs

# 4. Convert to a dictionary {class_id: weight_value}
class_weights_dict = class_weights_inv.to_dict()

print("Class distribution:\n", class_freqs)
print("\nInverse frequency weights:\n", class_weights_dict)

class_weights_dict = class_weights_inv.to_dict()

# 5. Save to a YAML file
with open('class_weights.yaml', 'w') as f:
    yaml.safe_dump(class_weights_dict, f, sort_keys=True)

Class distribution:
 labels
3     0.803690
2     0.085936
4     0.035772
6     0.029912
5     0.014811
7     0.011720
8     0.010400
1     0.003252
13    0.001288
15    0.000934
11    0.000869
14    0.000708
10    0.000515
9     0.000161
12    0.000032
Name: count, dtype: float64

Inverse frequency weights:
 {3: 1.2442610472336846, 2: 11.636568002997377, 4: 27.954995499549955, 6: 33.431646932185146, 5: 67.51739130434783, 7: 85.32417582417582, 8: 96.15479876160991, 1: 307.5049504950495, 13: 776.4499999999999, 15: 1070.9655172413793, 11: 1150.2962962962963, 14: 1411.7272727272727, 10: 1941.1249999999998, 9: 6211.599999999999, 12: 31057.999999999996}


In [5]:
from hydra.core.global_hydra import GlobalHydra
from tools.train import main
import torch
import hydra
import wandb

# data/data_FMO03_02_05/train

GlobalHydra.instance().clear()
hydra.initialize(config_path='../configs', job_name="dynamic_hydra")
cfg = hydra.compose(config_name="config_2024_12_20")
main(cfg)

wandb.finish()

/tmp/ipykernel_284355/1366617332.py:7: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  hydra.initialize(config_path='../configs', job_name="dynamic_hydra")


Setting the seed to 1
Building datasets ...
Connecting to Weights & Biases ...


wandb: Currently logged in as: karisu. Use `wandb login --relogin` to force relogin


Building the model ...
Preparing for training ...


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call,so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.